# Pure CNN VAE — Two Numbers → One Icon

**No word input. No class conditioning. Just a 2D latent vector.**

Train on the same 10,000 pre-generated black-and-white icons, then use only the decoder:

$$ (z_1,z_2) \longrightarrow \mathrm{Decoder} \longrightarrow \text{16×16 icon} $$

Run the cells from top to bottom. In Colab, a GPU runtime makes training faster.

**Names matter:** the encoder learns $\mu_1,\mu_2,\sigma_1,\sigma_2$ for each input image. It samples $z_i=\mu_i+\sigma_i\epsilon_i$. The two values you type directly into the decoder are **$z_1,z_2$**. Setting $z=\mu$ is also possible, but generation does not need an encoder or an input image.

Without a word condition, position in this one latent plane must represent both object identity and variation. Some positions can produce mixed or unclear shapes; a two-dimensional VAE is a deliberately small teaching model.

## 1. Download the existing training data

The images were prepared in advance. This cell never runs an image generator.

In [ ]:
# Download the pre-generated training data. No image generator runs here.
from pathlib import Path
import hashlib
import urllib.request

DATA_URL = 'https://raw.githubusercontent.com/dujing82-blip/tiny-black-white-generative-ai/main/data/icons_16x16.npz'
DATA_FILE = Path('icons_16x16.npz')
DATA_SHA256 = 'b3dc1a0b89c04a651b90de4f4f2cda78fa75c70befbc1e54ccc47ede87d40f52'

def checksum_matches(path):
    return path.exists() and hashlib.sha256(path.read_bytes()).hexdigest() == DATA_SHA256

# Reuse the local file when this cell is run again in the same Colab session.
if not checksum_matches(DATA_FILE):
    temporary = DATA_FILE.with_suffix('.download')
    urllib.request.urlretrieve(DATA_URL, temporary)
    if not checksum_matches(temporary):
        temporary.unlink(missing_ok=True)
        raise ValueError('Dataset checksum mismatch. Please rerun this cell.')
    temporary.replace(DATA_FILE)

print('Training dataset ready:', DATA_FILE, '(10,000 pre-generated icons)')


## 2. Load images only

The archive also stores class labels. We keep those solely for an optional plot after training; **the model never receives labels or words**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

keras.utils.set_random_seed(42)

with np.load(DATA_FILE, allow_pickle=False) as data:
    X = data['images'].astype('float32') / 255.0
    plot_labels = data['labels'].copy()   # Used ONLY to color the final scatter plot.
    plot_names = data['words'].tolist()

assert X.shape == (10000, 16, 16)
assert np.isin(X, [0.0, 1.0]).all()
X = X[..., np.newaxis]                   # CNN input: (10000, 16, 16, 1).
print('Training images:', X.shape)

# Show a random selection of the training images.
indices = np.random.default_rng(42).choice(len(X), 15, replace=False)
fig, axes = plt.subplots(3, 5, figsize=(8, 5))
for ax, index in zip(axes.flat, indices):
    ax.imshow(X[index, :, :, 0], cmap='gray', vmin=0, vmax=1, interpolation='nearest')
    ax.axis('off')
plt.tight_layout()
plt.show()


## 3. CNN encoder: an image → a distribution in 2D

The encoder outputs two means and two log-variances. Log-variance is convenient for training; $\sigma=\exp(\tfrac12\log\sigma^2)$.

Then we sample two latent coordinates:

$$z_1=\mu_1+\sigma_1\epsilon_1,\qquad z_2=\mu_2+\sigma_2\epsilon_2,\qquad \epsilon_i\sim\mathcal N(0,1).$$

In [ ]:
LATENT_DIM = 2

class Sampling(layers.Layer):
    def call(self, inputs):
        mean, log_var = inputs
        noise = tf.random.normal(tf.shape(mean))
        return mean + tf.exp(0.5 * log_var) * noise

# ONE input: image only. There is no word input or embedding.
image_input = keras.Input(shape=(16, 16, 1), name='image')
x = layers.Conv2D(32, 3, strides=2, padding='same', activation='relu')(image_input)
x = layers.Conv2D(64, 3, strides=2, padding='same', activation='relu')(x)
x = layers.Flatten()(x)
x = layers.Dense(128, activation='relu')(x)

z_mean = layers.Dense(2, name='z_mean')(x)
z_log_var = layers.Dense(2, name='z_log_var')(x)
z = Sampling()([z_mean, z_log_var])

encoder = keras.Model(image_input, [z_mean, z_log_var, z], name='pure_VAE_encoder')
encoder.summary()


## 4. CNN decoder: two numbers → an image

After training, this is all we need for generation. The same two coordinates always produce the same pixel probabilities in this decoder.

In [ ]:
# ONE input: the two latent coordinates. No word is supplied.
z_input = keras.Input(shape=(2,), name='z')
x = layers.Dense(4 * 4 * 64, activation='relu')(z_input)
x = layers.Reshape((4, 4, 64))(x)
x = layers.Conv2DTranspose(64, 3, strides=2, padding='same', activation='relu')(x)
x = layers.Conv2DTranspose(32, 3, strides=2, padding='same', activation='relu')(x)
output_image = layers.Conv2D(1, 3, padding='same', activation='sigmoid')(x)

decoder = keras.Model(z_input, output_image, name='pure_VAE_decoder')
decoder.summary()


## 5. Train the pure VAE

$$L=L_{\mathrm{reconstruction}}+\beta L_{\mathrm{KL}}.$$

Reconstruction teaches the decoder to draw. KL encourages the image-specific latent distributions to stay near the standard normal prior, so we can later generate by sampling $z\sim\mathcal N(0,I)$.

Here reconstruction is **summed over all 256 pixels**, and $\beta=1$ uses the standard VAE weighting. This does not guarantee every latent position produces a clean icon.

In [ ]:
class VAE(keras.Model):
    def __init__(self, encoder, decoder, beta=1.0):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.beta = beta
        # Track averages over the whole epoch, rather than just the last batch.
        self.loss_tracker = keras.metrics.Mean(name='loss')
        self.reconstruction_tracker = keras.metrics.Mean(name='reconstruction')
        self.kl_tracker = keras.metrics.Mean(name='KL')

    @property
    def metrics(self):
        return [self.loss_tracker, self.reconstruction_tracker, self.kl_tracker]

    def train_step(self, data):
        # This dataset contains ONLY images. No labels or condition inputs.
        images = data
        with tf.GradientTape() as tape:
            mean, log_var, z = self.encoder(images, training=True)
            reconstructed = self.decoder(z, training=True)

            # Binary crossentropy for every pixel; then sum pixels per image.
            pixel_loss = keras.losses.binary_crossentropy(images, reconstructed)
            reconstruction_loss = tf.reduce_mean(tf.reduce_sum(pixel_loss, axis=(1, 2)))

            # KL divergence from q(z|image) to the standard normal prior.
            kl_loss = -0.5 * tf.reduce_mean(tf.reduce_sum(
                1 + log_var - tf.square(mean) - tf.exp(log_var), axis=1))
            total_loss = reconstruction_loss + self.beta * kl_loss

        gradients = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(gradients, self.trainable_weights))
        batch_size = tf.cast(tf.shape(images)[0], tf.float32)
        self.loss_tracker.update_state(total_loss, sample_weight=batch_size)
        self.reconstruction_tracker.update_state(reconstruction_loss, sample_weight=batch_size)
        self.kl_tracker.update_state(kl_loss, sample_weight=batch_size)
        return {metric.name: metric.result() for metric in self.metrics}

# IMPORTANT: only X goes into the training dataset.
dataset = tf.data.Dataset.from_tensor_slices(X)
dataset = dataset.shuffle(len(X), seed=42).batch(128).prefetch(tf.data.AUTOTUNE)

BETA = 1.0
EPOCHS = 40
vae = VAE(encoder, decoder, beta=BETA)
vae.compile(optimizer=keras.optimizers.Adam(0.001))
history = vae.fit(dataset, epochs=EPOCHS)


## 6. Type two numbers and generate an icon

Start near −3 to +3. These axes are learned coordinates, not predefined controls such as width or object type. Very distant points may produce poor images. Thresholding is only for display, never for training.

In [ ]:
def generate(z1, z2, binary=True):
    point = np.array([[z1, z2]], dtype='float32')
    if not np.isfinite(point).all():
        raise ValueError('Please enter two finite numbers.')
    # Generation uses ONLY the decoder, with no input image and no word.
    probabilities = decoder(point, training=False).numpy()[0, :, :, 0]
    image = probabilities > 0.5 if binary else probabilities
    plt.figure(figsize=(3, 3))
    plt.imshow(image, cmap='gray', vmin=0, vmax=1, interpolation='nearest')
    plt.title(f'z1 = {z1:.2f}, z2 = {z2:.2f}')
    plt.axis('off')
    plt.show()
    return probabilities

# Change these two values and run again.
_ = generate(z1=0.5, z2=-1.0)


## 7. Interactive generator

Type your own coordinates, or press **Random z** to sample both coordinates from the standard normal distribution.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

z1_box = widgets.FloatText(value=0.5, description='z1:')
z2_box = widgets.FloatText(value=-1.0, description='z2:')
binary_box = widgets.Checkbox(value=True, description='Black/white display')
generate_button = widgets.Button(description='Generate icon')
random_button = widgets.Button(description='Random z')
output = widgets.Output()

def draw(_=None):
    with output:
        clear_output(wait=True)
        try:
            _ = generate(z1_box.value, z2_box.value, binary=binary_box.value)
        except ValueError as error:
            print(error)

def randomize(_):
    z1_box.value, z2_box.value = np.random.normal(size=2)
    draw()

generate_button.on_click(draw)
random_button.on_click(randomize)
display(widgets.VBox([widgets.HBox([z1_box, z2_box]), binary_box,
                      widgets.HBox([generate_button, random_button]), output]))
draw()


## 8. See the entire 2D latent plane

Each tile is decoded from one coordinate pair. Move horizontally to change z₁; move vertically to change z₂. A pure VAE uses one plane for all icon types.

In [ ]:
def show_latent_grid(limit=3.0, steps=15, binary=True):
    horizontal = np.linspace(-limit, limit, steps)
    vertical = np.linspace(limit, -limit, steps)  # Larger z2 at the top.
    points = np.array([[a, b] for b in vertical for a in horizontal], dtype='float32')
    images = decoder(points, training=False).numpy()[:, :, :, 0]
    if binary:
        images = images > 0.5
    # Arrange the generated images into a single tiled image.
    canvas = images.reshape(steps, steps, 16, 16).transpose(0, 2, 1, 3).reshape(steps*16, steps*16)
    # Tile centers correspond exactly to the sampled coordinates.
    half_step = (horizontal[1] - horizontal[0]) / 2
    edge = limit + half_step
    plt.figure(figsize=(9, 9))
    plt.imshow(canvas, cmap='gray', vmin=0, vmax=1, interpolation='nearest',
               extent=[-edge, edge, -edge, edge])
    plt.xlabel('z1')
    plt.ylabel('z2')
    plt.title('Pure VAE: two latent coordinates → icons')
    plt.show()

show_latent_grid()


## 9. Optional: what are $\mu$ and $\sigma$ for a real image?

The four distribution parameters depend on the image. Decode its mean with $z=\mu$ for a deterministic reconstruction, or sample around that mean for variations. A sampled reconstruction is not guaranteed to match the original exactly.

In [ ]:
image_index = 0   # Change to any index from 0 to 9999.
mean, log_var, sampled_z = encoder(X[image_index:image_index+1], training=False)
sigma = tf.exp(0.5 * log_var)
print('mu1, mu2:', mean.numpy()[0])
print('sigma1, sigma2:', sigma.numpy()[0])
print('sampled z1, z2:', sampled_z.numpy()[0])

mean_reconstruction = decoder(mean, training=False).numpy()[0, :, :, 0]
sampled_reconstruction = decoder(sampled_z, training=False).numpy()[0, :, :, 0]
fig, axes = plt.subplots(1, 3, figsize=(8, 3))
for ax, image, title in zip(axes,
        [X[image_index, :, :, 0], mean_reconstruction, sampled_reconstruction],
        ['Original', 'Decode z = mu', 'Decode sampled z']):
    ax.imshow(image, cmap='gray', vmin=0, vmax=1, interpolation='nearest')
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()


## 10. Optional: where do the training icons live?

Color the encoder means by their known classes **after training**. Colors help us interpret the plane; labels were never supplied to the VAE. The arrangement can change when the model is retrained.

In [ ]:
# Encode in batches to keep memory use small.
means, _, _ = encoder.predict(X, batch_size=256, verbose=0)
plt.figure(figsize=(7, 6))
for label, name in enumerate(plot_names):
    selected = means[plot_labels == label]
    plt.scatter(selected[:, 0], selected[:, 1], s=5, alpha=0.3, label=name)
plt.xlabel('mu1 (encoder mean on the z1 axis)')
plt.ylabel('mu2 (encoder mean on the z2 axis)')
plt.title('Training icons in the learned latent plane')
plt.legend()
plt.grid(alpha=0.2)
plt.show()


## Compare with the conditional VAE

| | Pure VAE (this notebook) | Conditional VAE |
|---|---|---|
| Encoder input | Image | Image + word |
| Decoder input | z | z + word |
| Where is object identity represented? | In the latent coordinates | The word supplies explicit class information |
| Can you request a particular class by word? | No | Yes |
| Generation needs an input image? | No | No |

**The VAE becomes a generator because, after learning, its decoder can turn a new latent point into an image.** The encoder is needed for training and encoding existing images; it is not needed when generating from two chosen coordinates.